# Graph learning on the SimPT-Power HV core
The HV core is a graph (3,787 buses; lines and transformers as edges) with a solved AC power flow for every 15-minute snapshot.
This notebook turns January 2026 (2,976 snapshots) into a supervised task: **predict every line's loading from the snapshot's national load and generation**, train on 1–21 January (hourly snapshots) and test on every 15-minute snapshot of 22–31 January.
It compares two baselines with a small graph neural network written in plain PyTorch. The point is the data pipeline; the model is deliberately simple.

In [ ]:
import os, json
from pathlib import Path
def find_data():
    """Locate the SimPT-Power open bundle: $SIMPT_DATA, a Kaggle input dataset, or the local build folder."""
    cands = [os.environ.get("SIMPT_DATA", "")]
    cands += [str(p.parent) for p in Path("/kaggle/input").rglob("bundle_manifest.json")] if Path("/kaggle/input").exists() else []
    cands += [f"{pre}output/simpt_power_release/open_bundle_{tag}" for tag in ("r3", "r2") for pre in ("../", "")]
    for c in cands:
        if c and (Path(c) / "bundle_manifest.json").exists():
            return Path(c)
    raise FileNotFoundError("SimPT-Power bundle not found; set SIMPT_DATA")
DATA = find_data()
manifest = json.loads((DATA / "bundle_manifest.json").read_text())
print(DATA, "|", manifest["bundle_of"])

In [ ]:
import duckdb, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
buses = pd.read_parquet(DATA / "hv/buses.parquet")
lines = pd.read_parquet(DATA / "hv/lines.parquet")
trafos = pd.read_parquet(DATA / "hv/transformers.parquet")
gens = pd.read_parquet(DATA / "hv/generators.parquet")
loads = pd.read_parquet(DATA / "hv/load_points.parquet")
bix = {b: i for i, b in enumerate(buses.bus_id)}
print(len(buses), "buses |", len(lines), "lines |", len(trafos), "transformers |", len(gens), "generators |", len(loads), "load points")

## 1. Graph tensors
Node features: voltage level (one-hot), degree, load-point flag and installed generation by technology. Edge features for lines: R, X, length and rating.

In [ ]:
kv_levels = [60, 130, 150, 220, 400]
X_kv = np.stack([(buses.voltage_kv == k).values for k in kv_levels], 1).astype(np.float32)
tech = gens.assign(t=gens.generation_source.fillna("other").str.lower()).groupby(["bus_id", "t"]).nameplate_mw.sum().unstack(fill_value=0)
top_tech = tech.sum().sort_values(ascending=False).index[:6]
X_gen = np.log1p(tech.reindex(buses.bus_id).fillna(0)[top_tech].values).astype(np.float32)
X_load = buses.bus_id.isin(loads.bus_id).values[:, None].astype(np.float32)
ln = lines[lines.from_bus.isin(bix) & lines.to_bus.isin(bix)].reset_index(drop=True)
src = np.r_[ln.from_bus.map(bix), trafos.hv_bus.map(bix)]; dst = np.r_[ln.to_bus.map(bix), trafos.lv_bus.map(bix)]
deg = np.bincount(np.r_[src, dst], minlength=len(buses))
X_static = np.hstack([X_kv, X_gen, X_load, np.log1p(deg)[:, None]]).astype(np.float32)
edge_index = torch.tensor(np.stack([np.r_[src, dst], np.r_[dst, src]]), dtype=torch.long)
E_attr = np.log1p(ln[["r_ohm_per_km", "x_ohm_per_km", "length_km", "max_i_ka"]].fillna(0).clip(lower=0).values).astype(np.float32)
print("node features", X_static.shape, "| edges (both directions)", edge_index.shape[1], "| tech columns:", list(top_tech))

## 2. Targets: line loading for 2,976 snapshots
Lines without a loading result (out of service or in an unsupplied island) are masked.

In [ ]:
snap = pd.read_parquet(DATA / "hv/snapshots_summary.parquet")
snap["t"] = pd.to_datetime(snap.timestamp_utc, utc=True)
jan = snap[(snap.t >= "2026-01-01") & (snap.t < "2026-02-01")].sort_values("t").reset_index(drop=True)
ll = duckdb.sql(f"select case_id, line_id, loading_percent from read_parquet('{DATA}/hv/line_loading_2026-01.parquet')").df()
Y = ll.pivot(index="case_id", columns="line_id", values="loading_percent").reindex(index=jan.case_id, columns=ln.line_id)
mask = Y.notna().values; Y = Y.fillna(0).values.astype(np.float32)
hour = jan.t.dt.hour + jan.t.dt.minute / 60
F = np.c_[jan.observed_load_mw / 1e4, jan.observed_generation_total_mw / 1e4, jan.observed_net_import_mw / 1e4,
          np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24), (jan.t.dt.weekday >= 5)].astype(np.float32)
train = (jan.t < "2026-01-22").values; test = ~train
print("Y", Y.shape, f"| train {train.sum()} / test {test.sum()} snapshots | lines with results: {mask.any(0).sum()}")

## 3. Baselines
(a) each line's training mean; (b) a separate linear regression per line on the six snapshot features (4,926 independent models).

In [ ]:
def mae(pred, sel):
    m = mask[sel]; return float(np.abs(pred - Y[sel])[m].mean())
mean_line = (Y[train] * mask[train]).sum(0) / np.maximum(mask[train].sum(0), 1)
A = np.c_[F, np.ones(len(F))]
W, *_ = np.linalg.lstsq(A[train], Y[train], rcond=None)
res = {"per-line mean": mae(np.tile(mean_line, (test.sum(), 1)), test), "per-line linear": mae(A[test] @ W, test)}
res

## 4. A small graph neural network
One shared model for all lines: the snapshot features are broadcast to every bus, concatenated with the static bus features, passed through three graph-convolution layers (mean aggregation over neighbours), and each line's value is read out from its two end buses and its own parameters.
Here the network learns the deviation of each line from its training mean, so it adds a shared, topology-aware correction to baseline (a).

In [ ]:
import torch.nn as nn
N = len(buses); ei = edge_index
deg_t = torch.bincount(ei[1], minlength=N).clamp(min=1).float()[:, None]
def propagate(h):          # mean over neighbours: h -> (B, N, d)
    out = torch.zeros_like(h); out.index_add_(1, ei[1], h[:, ei[0]]); return out / deg_t
class GNN(nn.Module):
    def __init__(self, fs, fd, fe, h=32):
        super().__init__()
        self.inp = nn.Linear(fs + fd, h)
        self.layers = nn.ModuleList([nn.Linear(2 * h, h) for _ in range(3)])
        self.out = nn.Sequential(nn.Linear(2 * h + fe + fd, h), nn.ReLU(), nn.Linear(h, 1))
    def forward(self, xs, f, ea, s, d):
        B = f.shape[0]
        h = torch.relu(self.inp(torch.cat([xs.expand(B, -1, -1), f[:, None].expand(-1, xs.shape[1], -1)], -1)))
        for lin in self.layers:
            h = h + torch.relu(lin(torch.cat([h, propagate(h)], -1)))
        hu, hv = h[:, s], h[:, d]
        z = torch.cat([hu + hv, (hu - hv).abs(), ea.expand(B, -1, -1), f[:, None].expand(-1, s.shape[0], -1)], -1)
        return self.out(z).squeeze(-1)
xs = torch.tensor(X_static)[None]; ea = torch.tensor(E_attr)[None]
s_t = torch.tensor(ln.from_bus.map(bix).values); d_t = torch.tensor(ln.to_bus.map(bix).values)
Ft, Yt, Mt = torch.tensor(F), torch.tensor(Y), torch.tensor(mask)
base_t = torch.tensor(mean_line, dtype=torch.float32)
model = GNN(X_static.shape[1], F.shape[1], E_attr.shape[1])
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
idx_tr = np.flatnonzero(train)[::4]          # hourly training snapshots keep the CPU run to a few minutes
for epoch in range(25):
    np.random.shuffle(idx_tr); tot = 0
    for k in range(0, len(idx_tr), 64):
        b = idx_tr[k:k + 64]
        p = model(xs, Ft[b], ea, s_t, d_t) + base_t
        loss = (p - Yt[b]).abs()[Mt[b]].mean()
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(b)
    if epoch % 5 == 4: print(f"epoch {epoch+1}: train MAE {tot / len(idx_tr):.2f} pp")
with torch.no_grad():
    pred = torch.cat([model(xs, Ft[np.flatnonzero(test)[k:k+64]], ea, s_t, d_t) + base_t for k in range(0, test.sum(), 64)]).numpy()
res["per-line mean + GNN correction"] = mae(pred, test)
pd.Series(res, name="test MAE (percentage points)").round(2)

On lines with a training history, the 4,926 separate linear regressions remain the strongest model; the shared GNN improves on the per-line mean with a single set of weights. Its use is for lines without history, tested next.

## 5. Generalising to unseen lines
Hide 20% of the lines during training and evaluate only on them. No per-line statistic exists for these lines, so the reference is the global mean of the other lines; the GNN is now trained on the loading itself and predicts it from the graph alone.

In [ ]:
rng = np.random.default_rng(0)
hidden = rng.random(len(ln)) < 0.2
Mt_tr = Mt.clone(); Mt_tr[:, torch.tensor(hidden)] = False
model2 = GNN(X_static.shape[1], F.shape[1], E_attr.shape[1]); opt = torch.optim.Adam(model2.parameters(), lr=3e-3)
for epoch in range(25):
    np.random.shuffle(idx_tr)
    for k in range(0, len(idx_tr), 64):
        b = idx_tr[k:k + 64]
        loss = (model2(xs, Ft[b], ea, s_t, d_t) - Yt[b]).abs()[Mt_tr[b]].mean()
        opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    pred2 = torch.cat([model2(xs, Ft[np.flatnonzero(test)[k:k+64]], ea, s_t, d_t) for k in range(0, test.sum(), 64)]).numpy()
m = mask[test][:, hidden]
glob = float(Y[train][:, ~hidden][mask[train][:, ~hidden]].mean())
print(f"hidden lines: {hidden.sum()} | test MAE on them: global mean {np.abs(glob - Y[test][:, hidden])[m].mean():.2f} pp, "
      f"GNN {np.abs(pred2[:, hidden] - Y[test][:, hidden])[m].mean():.2f} pp")

### Where to go from here
* Use all 31,492 snapshots: the release ships one compressed DuckDB per month (`hv_core/`), with bus voltages and line loadings.
* Add per-bus injections as node features (loads and dispatch are allocation rules in `operating.resource_timeseries_definition` of the main database).
* Topology changes (N-1, see notebook 2) give natural graph-level augmentations.

**Citation.** SimPT-Power is a simulation database built from public data. It is *not* the operator's network or a digital twin: converged power flows show numerical consistency, not agreement with operator state estimates. Please cite the archived release (Zenodo DOI) and the data paper. Data licences follow the sources (E-REDES CC BY 4.0, OpenStreetMap-derived layers ODbL); see `SOURCE_LICENSES.md` in the release.